In [1]:
!pip install nltk
!pip install numpy pandas
!pip install scikit-learn
!pip install gensim
!pip install matplotlib

import nltk
import numpy as np
import pandas as pd
import sklearn
import gensim
import matplotlib.pyplot as plt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# AIG230 Natural Language Processing - Assignment 5

## Corpus Choice
For this assignment, I selected Option A from the Gutenberg corpus and used austen-emma.txt.

## Part A - Text Preprocessing

In [ ]:
from nltk.corpus import gutenberg
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
raw_text = gutenberg.raw('austen-emma.txt')

print(f"Total characters: {len(raw_text)}")

tokens_before = nltk.word_tokenize(raw_text)
print("Total tokens BEFORE preprocessing:", len(tokens_before))


[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/mahboobehyasini/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mahboobehyasini/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mahboobehyasini/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total characters: 887071
Total tokens BEFORE preprocessing: 191855


In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))
def preprocess(text: str, remove_stopwords=True):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

tokens_after = preprocess(raw_text,remove_stopwords=True)
print(tokens_after[:50])

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mahboobehyasini/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/mahboobehyasini/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


['emma', 'jane', 'austen', 'volume', 'chapter', 'emma', 'woodhouse', 'handsome', 'clever', 'rich', 'comfortable', 'home', 'happy', 'disposition', 'seemed', 'unite', 'best', 'blessing', 'existence', 'lived', 'nearly', 'year', 'world', 'little', 'distress', 'vex', 'youngest', 'two', 'daughter', 'affectionate', 'indulgent', 'father', 'consequence', 'sister', 'marriage', 'mistress', 'house', 'early', 'period', 'mother', 'died', 'long', 'ago', 'indistinct', 'remembrance', 'caress', 'place', 'supplied', 'excellent', 'woman']


In [ ]:
print("Total tokens AFTER preprocessing:", len(tokens_after))

vocab = set(tokens_after)
print("Vocabulary size:", len(vocab))

from collections import Counter
freq = Counter(tokens_after)
print('Top 20 most frequent tokens:', freq.most_common(20))

Total tokens AFTER preprocessing: 69689
Vocabulary size: 6140
Top 20 most frequent tokens: [('emma', 860), ('could', 836), ('would', 818), ('miss', 600), ('must', 566), ('harriet', 500), ('much', 484), ('said', 483), ('thing', 456), ('one', 451), ('weston', 445), ('every', 435), ('think', 406), ('elton', 383), ('knightley', 379), ('well', 378), ('know', 365), ('little', 359), ('never', 358), ('say', 341)]


## A2 Choice Justification

I removed stopwords in Parts A and B because very common words (like the, and, and is) appear too often and can make the important words harder to see. I chose lemmatization instead of stemming because lemmatization keeps proper base words and usually preserves meaning better in literary text.

## A3 Reflection

My preprocessing choices affect all later tasks. Lowercasing and punctuation removal reduce noise and make the vocabulary more consistent. Stopword removal helps Bag-of-Words and TF-IDF focus on informative terms, which makes topic interpretation and document comparison clearer. Lemmatization combines related word forms, so sparsity is reduced and token statistics are more stable. For language modeling, if preprocessing is too aggressive, useful grammatical context can be lost. For embeddings, richer context can improve semantic neighborhoods because Word2Vec learns from surrounding words. Overall, preprocessing should be adjusted based on the goal of each task.

## Part B - Text Representation

In [ ]:
def chunk_tokens(tokens, chunk_size=500):
    return [' '.join(tokens[i:i+chunk_size])
            for i in range(0, len(tokens), chunk_size)]

documents = chunk_tokens(tokens_after, 500)

print("Number of documents:", len(documents))

Number of documents: 140


### B1 Strategy Justification

I used fixed-size chunks of 500 tokens for each document.

Reason:
- This gives enough documents for meaningful similarity comparison.
- Each chunk is still long enough to keep local context.
- A size of 500 is in the required 500-1000 range and gives a good balance between detail and stability.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words='english')
X_bow = count_vectorizer.fit_transform(documents)

print("Shape of Bag-of-Words matrix:", X_bow.shape)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(documents)

print("Shape of TF-IDF matrix:", X_tfidf.shape)

feature_names = tfidf_vectorizer.get_feature_names_out()

doc_ids = [0, 10]

for doc_id in doc_ids:
    row = X_tfidf[doc_id].toarray().flatten()
    top_indices = np.argsort(row)[-15:]

    print(f"\nTop 15 TF-IDF terms for document {doc_id}:")
    for index in reversed(top_indices):
        print(feature_names[index])

Shape of Bag-of-Words matrix: (140, 5983)
Shape of TF-IDF matrix: (140, 5983)

Top 15 TF-IDF terms for document 0:
taylor
year
governess
sixteen
miss
hating
gentle
emma
sorrow
mile
change
father
affection
friend
matrimony

Top 15 TF-IDF terms for document 10:
emma
figure
plague
health
advice
weston
knightley
bloom
smith
shall
love
harriet
sister
seldom
liberty


### B2 Interpretation of Top TF-IDF Terms

In document 0, terms like taylor, governess, sorrow, friend, and matrimony are among the highest TF-IDF values. These words match the early relationships and setting, so the model gives them stronger document-specific weight.

In document 10, terms such as figure, health, advice, weston, knightley, and harriet are more prominent. This suggests that this chunk focuses on a different scene with different characters and themes.

Overall, this shows TF-IDF is working as expected because it highlights terms that are distinctive in each document, not only words that are frequent in the full corpus.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_tfidf)

print("Similarity matrix shape:", similarity_matrix.shape)
import numpy as np

np.fill_diagonal(similarity_matrix, 0)

max_index = np.unravel_index(
    np.argmax(similarity_matrix),
    similarity_matrix.shape
)

doc1, doc2 = max_index
max_score = similarity_matrix[doc1, doc2]

print("Most similar documents:", doc1, "and", doc2)
print("Similarity score:", max_score)

import pandas as pd

small_table = pd.DataFrame(
    similarity_matrix[:5, :5],
    columns=[f"Doc{i}" for i in range(5)],
    index=[f"Doc{i}" for i in range(5)]
)

print(small_table)

Similarity matrix shape: (140, 140)
Most similar documents: 48 and 49
Similarity score: 0.46082622193482403
          Doc0      Doc1      Doc2      Doc3      Doc4
Doc0  0.000000  0.259444  0.283286  0.210043  0.257242
Doc1  0.259444  0.000000  0.325234  0.138660  0.233211
Doc2  0.283286  0.325234  0.000000  0.199496  0.205559
Doc3  0.210043  0.138660  0.199496  0.000000  0.199873
Doc4  0.257242  0.233211  0.205559  0.199873  0.000000


### B3 Surprising Similarity

The most similar pair is Documents 48 and 49, with cosine similarity 0.4608. This is reasonable because nearby chunks can share character names, dialogue style, and repeated vocabulary. Even though they are separate chunks, they likely represent closely related narrative moments.

## Part C - Word Embeddings

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

def preprocess_for_embeddings(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

sentences = sent_tokenize(raw_text)
tokenized_sentences = [preprocess_for_embeddings(sent) for sent in sentences]

print("Number of sentences:", len(tokenized_sentences))
print("First tokenized sentence:", tokenized_sentences[0])

Number of sentences: 7493
First tokenized sentence: ['emma', 'jane', 'austen', 'volume', 'chapter', 'emma', 'woodhouse', 'handsome', 'clever', 'rich', 'comfortable', 'home', 'happy', 'disposition', 'seemed', 'unite', 'best', 'blessing', 'existence', 'lived', 'nearly', 'year', 'world', 'little', 'distress', 'vex']


In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=0,
    epochs=10
)


### C2 Hyperparameter Justification

I selected vector_size = 100, window = 5, min_count = 3, sg = 0, and epochs = 10.

Reason:
- vector_size = 100 gives enough detail without being too large for one novel.
- window = 5 captures nearby context in sentence-level text.
- min_count = 3 removes very rare words and reduces noise.
- sg = 0 (CBOW) is usually more stable on smaller datasets.
- epochs = 10 gives the model enough training passes for more consistent vectors.

These settings are suitable for this corpus size and produced understandable similarity results.

In [ ]:
target_words = ['emma', 'harriet', 'mr', 'knightley', 'woodhouse']

for word in target_words:
    print(f"\nTop 10 words similar to '{word}':")
    try:
        similar_words = model.wv.most_similar(word, topn=10)
        for w, score in similar_words:
            print(f"{w}: {score:.3f}")
    except KeyError:
        print(f"'{word}' not in vocabulary!")


Top 10 words similar to 'emma':
smiling: 0.997
minute: 0.997
listened: 0.997
dine: 0.997
sat: 0.997
longer: 0.997
instead: 0.997
passed: 0.997
intimacy: 0.997
went: 0.997

Top 10 words similar to 'harriet':
sat: 0.998
brother: 0.997
usual: 0.997
particularly: 0.997
looked: 0.997
added: 0.997
appeared: 0.997
immediately: 0.997
grateful: 0.997
sitting: 0.997

Top 10 words similar to 'mr':
talking: 0.999
wonder: 0.999
person: 0.999
remarkably: 0.999
speaking: 0.999
compliment: 0.999
thousand: 0.999
character: 0.999
charade: 0.999
fancy: 0.999

Top 10 words similar to 'knightley':
henry: 0.995
isabella: 0.995
smiling: 0.995
sat: 0.995
listened: 0.995
warmly: 0.995
emma: 0.995
talked: 0.995
pianoforte: 0.995
return: 0.995

Top 10 words similar to 'woodhouse':
bates: 0.998
taylor: 0.981
smith: 0.978
fairfax: 0.977
nash: 0.956
hawkins: 0.900
dear: 0.894
bickerton: 0.875
otway: 0.871
poor: 0.858


### C3 Short Interpretation

Most nearest-neighbor results are reasonable for this corpus. For example, emma is close to knightley, harriet, weston, and elton, which are characters that often appear in related contexts. Harriet and knightley also return character-centered neighbors. The word mr is less precise because it is very frequent and appears in many contexts. Overall, the model captures useful semantic patterns, with some expected noise because training data comes from only one book.

In [ ]:
analogy_queries = [
    ('emma', 'harriet', 'isabella'),
    ('weston', 'harriet', 'emma'),
    ('mr', 'weston', 'knightley')
]

for w1, w2, w3 in analogy_queries:
    print(f"\nAnalogy: {w1} - {w2} + {w3} ≈ ?")
    try:
        results = model.wv.most_similar(positive=[w1, w3], negative=[w2], topn=5)
        for word, score in results:
            print(f"{word}: {score:.3f}")
    except KeyError as e:
        print(f"Word not in vocabulary: {e}")



Analogy: emma - harriet + isabella ≈ ?
trying: 0.996
perry: 0.996
took: 0.996
mother: 0.996
charge: 0.996

Analogy: weston - harriet + emma ≈ ?
coming: 0.994
turned: 0.994
opinion: 0.994
morning: 0.994
seeing: 0.994

Analogy: mr - weston + knightley ≈ ?
isabella: 0.993
henry: 0.993
believe: 0.993
talking: 0.993
shew: 0.993


### C4 Analogy Comment

The analogy results are weaker, which is expected for a small corpus. Since the model is trained on only one novel, it has fewer examples of reliable relational patterns than large general corpora. Because of this, some analogy outputs are inconsistent.